In [ ]:
!pip install plotly pandas requests scipy statsmodels nbformat -q
import requests, pandas as pd, numpy as np
import plotly.graph_objects as go
import plotly.express as px
from scipy import stats
from statsmodels.tsa.stattools import adfuller

API_URL = 'https://djtnqbvkhqftmtnsityx.supabase.co/rest/v1/'
API_KEY = 'sb_publishable_NnOjc1iJWmA7j418h79mEg_AHh9h49u'
HEADERS = {'apikey': API_KEY, 'Authorization': f'Bearer {API_KEY}'}

def query(endpoint, select='*', filters=None, limit=50000):
    params = {'select': select, 'limit': limit}
    if filters: params.update({k: f'eq.{v}' for k,v in filters.items()})
    r = requests.get(f'{API_URL}{endpoint}', headers=HEADERS, params=params)
    return pd.DataFrame(r.json()) if r.status_code == 200 else pd.DataFrame()

# Funding Rate Research Report
## Crypto + Equity Perpetual Futures Analysis

**Data**: 519K+ funding rate observations across 5 venues and 21 symbols  
**Period**: Apr 2019 – Jun 2026 (crypto), Nov 2025 – Jun 2026 (equity)  
**Venues**: Binance, Hyperliquid, Deribit, Binance TradFi, Hyperliquid xyz  

This report tests 3 key hypotheses about perpetual futures funding rates:
1. Weekend Oracle Freeze (equity perps)
2. Cross-Venue Arbitrage Efficiency
3. Funding Rate Mean Reversion

In [ ]:
df = query('daily_funding', select='*')
equity = df[df['asset_class'] == 'equity']
crypto = df[df['asset_class'] == 'crypto']
spreads = query('venue_comparison', select='*', limit=50000)
print(f'Total: {len(df):,} rows | Crypto: {len(crypto):,} | Equity: {len(equity):,}')
print(f'Date range: {df.date.min()} to {df.date.max()}')

## H1: Weekend Oracle Freeze

**Null (H₀)**: Equity perp funding rates are the same on weekends and weekdays.  
**Alternative (H₁)**: Equity perp funding rates are significantly different on weekends due to oracle freeze.

Since equity perps reference spot indices that stop updating on Friday close,  
but the perp continues trading on sentiment, we expect different funding behavior.

In [ ]:
# Split equity data
eq_weekend = equity[equity['is_weekend'] == True]['avg_rate_bps'].dropna()
eq_weekday = equity[equity['is_weekend'] == False]['avg_rate_bps'].dropna()

# Control group: crypto perps (no oracle freeze)
cr_weekend = crypto[crypto['is_weekend'] == True]['avg_rate_bps'].dropna()
cr_weekday = crypto[crypto['is_weekend'] == False]['avg_rate_bps'].dropna()

# Welch's t-test (unequal variance)
t_eq, p_eq = stats.ttest_ind(eq_weekend, eq_weekday, equal_var=False)
t_cr, p_cr = stats.ttest_ind(cr_weekend, cr_weekday, equal_var=False)

# Cohen's d effect size
d_eq = (eq_weekend.mean() - eq_weekday.mean()) / np.sqrt((eq_weekend.var() + eq_weekday.var()) / 2)
d_cr = (cr_weekend.mean() - cr_weekday.mean()) / np.sqrt((cr_weekend.var() + cr_weekday.var()) / 2)

print(f'=== Equity Perps ===')
print(f'Weekend mean: {eq_weekend.mean():.1f} bps | Weekday mean: {eq_weekday.mean():.1f} bps')
print(f't-statistic: {t_eq:.3f} | p-value: {p_eq:.6f}')
print(f"Cohen's d: {d_eq:.3f} ({'large' if abs(d_eq)>0.8 else 'medium' if abs(d_eq)>0.5 else 'small'} effect)")
print(f'Significant: {"YES ✅" if p_eq < 0.05 else "NO — cannot reject H₀"}')
print(f'\n=== Crypto Perps (Control) ===')
print(f'Weekend mean: {cr_weekend.mean():.1f} bps | Weekday mean: {cr_weekday.mean():.1f} bps')
print(f't-statistic: {t_cr:.3f} | p-value: {p_cr:.6f}')
print(f"Cohen's d: {d_cr:.3f}")
print(f'Significant: {"YES" if p_cr < 0.05 else "NO — crypto perps behave consistently ✅"}')

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(rows=1, cols=2, subplot_titles=('Equity Perps (Oracle Freeze)', 'Crypto Perps (Control)'))

for asset, col, data in [('Equity', 1, equity), ('Crypto', 2, crypto)]:
    for is_wknd, label, color in [(True, 'Weekend', 'red'), (False, 'Weekday', 'blue')]:
        subset = data[data['is_weekend'] == is_wknd]
        fig.add_trace(go.Box(y=subset['avg_rate_bps'], name=f'{label}', marker_color=color, showlegend=(col==1)), row=1, col=col)

fig.update_layout(title=f'H1: Weekend Oracle Freeze — Equity vs Crypto Funding Rates<br>Equity p={p_eq:.4f}, d={d_eq:.2f} | Crypto p={p_cr:.4f}, d={d_cr:.2f}', height=500)
fig.show()

## H2: Cross-Venue Arbitrage Efficiency

**Null (H₀)**: Cross-venue funding rate spreads are within transaction costs (≤10 bps).  
**Alternative (H₁)**: Statistically significant arbitrage opportunities exist (spreads > 10 bps).

We test whether venue spreads exceed a 10 bps transaction cost threshold.

In [ ]:
spreads_clean = spreads[spreads['max_cross_spread_bps'].notna() & (spreads['max_cross_spread_bps'] > 0)]
print(f'Cross-venue comparison rows: {len(spreads):,}')
print(f'Rows with positive spread: {len(spreads_clean):,} ({len(spreads_clean)/len(spreads)*100:.1f}%)')

# One-sample t-test: is mean spread > 10 bps?
threshold = 10
t_stat, p_val = stats.ttest_1samp(spreads_clean['max_cross_spread_bps'], threshold)
# One-tailed: divide p by 2 since we're testing > threshold
p_one_tailed = p_val / 2 if t_stat > 0 else 1 - p_val / 2

print(f'\nMean spread: {spreads_clean.max_cross_spread_bps.mean():.1f} bps')
print(f'Median spread: {spreads_clean.max_cross_spread_bps.median():.1f} bps')
print(f'95th percentile: {spreads_clean.max_cross_spread_bps.quantile(0.95):.1f} bps')
print(f'Max spread: {spreads_clean.max_cross_spread_bps.max():.1f} bps')
print(f'\nt-statistic (vs {threshold} bps): {t_stat:.3f}')
print(f'p-value (one-tailed): {p_one_tailed:.6f}')
print(f'Significant: {"YES ✅ — spreads exceed transaction costs" if p_one_tailed < 0.05 else "NO"}')

In [ ]:
fig = go.Figure()
fig.add_trace(go.Histogram(x=spreads_clean['max_cross_spread_bps'], nbinsx=50, name='Spread Distribution'))
fig.add_vline(x=10, line_dash='dash', line_color='red', annotation_text='Cost Threshold (10 bps)')
fig.add_vline(x=spreads_clean['max_cross_spread_bps'].mean(), line_dash='dash', line_color='green', annotation_text=f'Mean ({spreads_clean.max_cross_spread_bps.mean():.0f} bps)')
fig.update_layout(title=f'H2: Cross-Venue Spread Distribution<br>Mean={spreads_clean.max_cross_spread_bps.mean():.0f} bps, p={p_one_tailed:.4f}', xaxis_title='Spread (bps)', height=400)
fig.show()

## H3: Funding Rate Mean Reversion

**Null (H₀)**: Funding rates follow a random walk (unit root present).  
**Alternative (H₁)**: Funding rates are mean-reverting (stationary).

We use the Augmented Dickey-Fuller test. Rejecting H₀ means rates revert to a mean.

In [ ]:
# Get daily funding for Binance BTC (most liquid, longest history)
btc = df[(df['symbol'] == 'BTCUSDT') & (df['venue'] == 'binance')].sort_values('date')
rates = btc['avg_rate_bps'].dropna().values

# ADF test
adf_result = adfuller(rates, maxlag=30)
print(f'ADF Statistic: {adf_result[0]:.3f}')
print(f'p-value: {adf_result[1]:.6f}')
print(f'Critical values: {adf_result[4]}')
print(f'Lags used: {adf_result[2]}')
print(f'Mean-reverting: {"YES ✅ — rates revert to mean" if adf_result[1] < 0.05 else "NO — cannot reject random walk"}')

# Also test on equity perps
eq_rates = equity[equity['symbol'].str.contains('SPY')]['avg_rate_bps'].dropna().values[:200]
if len(eq_rates) > 30:
    adf_eq = adfuller(eq_rates, maxlag=10)
    print(f'\nEquity (SPY) ADF: {adf_eq[0]:.3f}, p={adf_eq[1]:.4f}')

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(y=rates[-365:], mode='lines', name='BTC Funding Rate'))
fig.add_hline(y=rates.mean(), line_dash='dash', line_color='red', annotation_text=f'Mean: {rates.mean():.0f} bps')
fig.update_layout(title=f'H3: BTCUSDT Funding Rate — Last 365 Days<br>ADF p={adf_result[1]:.4f} → {"Mean-Reverting ✅" if adf_result[1] < 0.05 else "Random Walk"}', height=400)
fig.show()

## Limitations

1. **Short equity perp history**: Only ~8 months of data (Nov 2025 – Jun 2026). Weekend tests have limited power.
2. **Survivorship bias**: Only venues with available APIs are included. Delisted perps are excluded.
3. **Transaction costs**: Cross-venue arb analysis uses a fixed 10 bps threshold. Actual costs vary by venue and size.
4. **No execution modeling**: The analysis identifies opportunities but does not model execution feasibility.
5. **Single exchange concentration**: Binance dominates the dataset. Hyperliquid and Deribit provide useful but smaller samples.

## Conclusion

- **H1 (Weekend Oracle Freeze)**: [PENDING] — reported from statistical test results above
- **H2 (Cross-Venue Arbitrage)**: [PENDING] — reported from statistical test results above  
- **H3 (Mean Reversion)**: [PENDING] — reported from statistical test results above